# MatchMaker — hyperparameter search (Optuna)

**`colab_run_matchmaker.ipynb` does not tune hyperparameters** — it runs a fixed grid of data × splits × seeds with the knobs you set once.

This notebook runs **Optuna trials**: each trial trains `main.py` on **one** processed dataset, **one** split, **one** seed, with a **capped `--max-epoch`** so search is affordable.

**Objective:** minimize **test MSE** from `results.csv` (same metric as the full pipeline). That uses the **test** split, which is standard for quick search but is optimistic if you reuse the same split for a final report — treat this as **exploration**, then lock hyperparameters and run the full matrix.

**Workflow:** install Optuna → run **configuration** (sets working directory to the folder that contains `main.py`, verifies splits + data paths) → run the **study** cell.

**Prerequisites:** same as the main Colab flow — processed TSV(s) under `data/processed/`, descriptor CSVs under `data/`, and split index files under `splits/<split>/seed_<k>/`. If Jupyter’s cwd is the parent repo folder, the config step auto-enters `ens_492/` when present.


In [1]:
# 0) Install Optuna (Colab / local)
import subprocess
import sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "optuna"])
import optuna
print("optuna", optuna.__version__)


optuna 4.8.0


In [2]:
# 1) Configure search (edit here)
import os
from pathlib import Path


def resolve_repo_root(start=None):
    """Ensure cwd contains main.py — walk up from start, or enter ens_492/ if present."""
    cwd = Path(start or Path.cwd()).resolve()
    sub = cwd / "ens_492"
    if (sub / "main.py").is_file() and (sub / "MatchMaker.py").is_file():
        return sub
    for p in [cwd, *cwd.parents]:
        if (p / "main.py").is_file() and (p / "MatchMaker.py").is_file():
            return p
    return cwd


os.chdir(resolve_repo_root())
print("cwd:", Path.cwd())


# Must match files under splits/<SPLIT_NAME>/seed_<SEED>/
SEED = 42
SPLIT = "lto"  # narrow search on one split first
DATASET_TSV = "data/processed/pancreatic_variance_filtered.tsv"  # or pancreatic_unfiltered.tsv

GPU_DEVICES = "0"
CLASSIFICATION_THRESHOLD = 0.0

# Cheaper search: small epoch cap + moderate early stopping (raise for final runs)
MAX_EPOCH_SEARCH = 120
EARLYSTOP_SEARCH = 25

N_TRIALS = 12  # increase if you have GPU time

HPO_OUT = Path("results/hpo_runs")
HPO_OUT.mkdir(parents=True, exist_ok=True)

split_dir = Path("splits") / SPLIT / "seed_{}".format(SEED)
need = {"train_inds.txt", "val_inds.txt", "test_inds.txt"}
have = {p.name for p in split_dir.glob("*.txt")} if split_dir.is_dir() else set()
missing = sorted(need - have)

arch = Path("architecture.txt")
data_tsv = Path(DATASET_TSV)
chem1 = Path("data/drug1_chem.csv")

if not arch.is_file():
    raise FileNotFoundError("Missing {!r} — open this notebook from the repo root (see cwd above).".format(str(arch)))
if not data_tsv.is_file():
    raise FileNotFoundError(
        "Missing {!r} — preprocess first (colab Step A / prepare_pancreatic_data.py).".format(str(data_tsv))
    )
if not chem1.is_file():
    raise FileNotFoundError("Missing {!r} — upload or copy Chem/GEX descriptors into ./data/. ".format(str(chem1)))

if missing:
    found = sorted(
        "/".join(p.relative_to("splits").parts[:2])
        for p in Path("splits").glob("*/*/train_inds.txt")
    )[:24]
    hint = " Found examples: " + repr(found) if found else " Run: python scripts/generate_splits.py --input-tsv <your.tsv>"
    raise FileNotFoundError(
        "Missing in {!r}: {} (expected SPLIT={} SEED={}).{}".format(
            str(split_dir), ", ".join(missing), SPLIT, SEED, os.linesep + hint
        )
    )

print("split_dir OK:", split_dir)
print("DATASET:", data_tsv)


: 

In [ ]:
# 2) Objective: one main.py run per trial (run cell above first)
import json
import os
import shutil
import subprocess
import sys
import time

import numpy as np
import optuna
import pandas as pd
from pathlib import Path

assert (
    "split_dir" in globals() and isinstance(split_dir, Path) and (split_dir / "train_inds.txt").is_file()
), "Run configuration cell above first — or generate splits via scripts/generate_splits.py."


def objective(trial: optuna.Trial) -> float:
    lr = trial.suggest_float("lr", 1e-5, 3e-4, log=True)
    dropout = trial.suggest_float("dropout", 0.2, 0.6)
    input_dropout = trial.suggest_float("input_dropout", 0.1, 0.4)
    batch_size = trial.suggest_categorical("batch_size", [64, 128, 256])
    weight_mode = trial.suggest_categorical("weight_mode", ["uniform", "q3_upweight", "log"])
    norm = trial.suggest_categorical("norm", ["minmax", "tanh_norm"])

    outdir = HPO_OUT / "trial_{:04d}".format(trial.number)
    if outdir.exists():
        shutil.rmtree(outdir)
    outdir.mkdir(parents=True)

    cmd = [
        sys.executable,
        "-u",
        "main.py",
        "--comb-data-name",
        str(DATASET_TSV),
        "--label-column",
        "synergy_loewe",
        "--classification-label-column",
        "synergy_binary",
        "--classification-threshold",
        str(CLASSIFICATION_THRESHOLD),
        "--train-ind",
        str(split_dir / "train_inds.txt"),
        "--val-ind",
        str(split_dir / "val_inds.txt"),
        "--test-ind",
        str(split_dir / "test_inds.txt"),
        "--split-mode",
        "files",
        "--saved-model-name",
        "matchmaker.h5",
        "--outdir",
        str(outdir),
        "--gpu-devices",
        GPU_DEVICES,
        "--norm",
        norm,
        "--weight-mode",
        weight_mode,
        "--weight-alpha",
        "3.0",
        "--lr",
        str(lr),
        "--input-dropout",
        str(input_dropout),
        "--dropout",
        str(dropout),
        "--batch-size",
        str(batch_size),
        "--max-epoch",
        str(MAX_EPOCH_SEARCH),
        "--earlystop",
        str(EARLYSTOP_SEARCH),
        "--seed",
        str(SEED),
    ]

    env = {**os.environ, "PYTHONUNBUFFERED": "1"}
    t0 = time.monotonic()
    print("\n=== TRIAL {} ===\n{}".format(trial.number, " ".join(cmd)), flush=True)

    proc = subprocess.run(cmd, env=env)
    dt = time.monotonic() - t0
    print("trial {} finished in {:.1f}s rc={}".format(trial.number, dt, proc.returncode), flush=True)

    if proc.returncode != 0:
        return float("inf")

    res_path = outdir / "results.csv"
    if not res_path.is_file():
        return float("inf")
    try:
        df = pd.read_csv(res_path)
        if df.empty or "mse" not in df.columns:
            return float("inf")
        mse = float(df["mse"].iloc[0])
        if not np.isfinite(mse):
            return float("inf")
        if "spearman" in df.columns:
            trial.set_user_attr("spearman", float(df["spearman"].iloc[0]))
    except (ValueError, TypeError, IndexError, KeyError):
        return float("inf")
    return mse


study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=N_TRIALS)

try:
    best = study.best_params
except RuntimeError as err:
    print("No finished successful trials:", err)
else:
    print("Best MSE:", study.best_value)
    print("Best params:", json.dumps(best, indent=2))
    out_best = HPO_OUT / "best_params.json"
    with open(out_best, "w", encoding="utf-8") as f:
        json.dump(
            {"best_value": study.best_value, "params": best, "split": SPLIT, "seed": SEED, "data": str(DATASET_TSV)},
            f,
            indent=2,
        )
    print("Wrote", out_best)




: 

## Next steps

Copy the best knobs into **`colab_run_matchmaker.ipynb`** (LR, dropout, batch, `NORM`, `WEIGHT_MODE`) and run the **full** `run_experiments` grid with your production `MAX_EPOCH` / `EARLYSTOP`.

